In [65]:
import geopandas as gpd
import joblib
import numpy as np
import odc.geo  # noqa: F401
import pandas as pd
from shapely import geometry
from sklearn.ensemble import RandomForestClassifier

from utils import load_data, safe_divide

In [66]:
%reload_ext autoreload
%autoreload 2

## Find and load S2 data

Load data and set up your array to use for prediction

In [67]:
# Configure some things up front
chunks = dict(x=256, y=256)
datetime = "2022"

bbox_fiji = [177.14, -18.41, 178.80, -17.2]
bbox = [178.14, -17.81, 178.50, -17.4]
bbox_geometry = geometry.box(*bbox)

gdf = gpd.GeoDataFrame({'geometry': [bbox_geometry]}, crs='EPSG:4326')
gdf.explore()

In [ ]:
merged = load_data(bbox, chunks=dict(x=256, y=256), resolution=10)

In [ ]:
merged

## Train and predict

When you change your training data, you can re-train and predict here.

In [ ]:
training_file = "training_data/mred-tdata.geojson"

tdata = gpd.read_file(training_file, bbox=bbox_geometry)
tdata.explore()

In [ ]:
len(tdata)

In [ ]:
# Get values for each of the image bands at each of the points.
pts_proj = tdata.to_crs(merged.odc.crs)

# a DataArray with x & y coords
pts_da = pts_proj.assign(x=pts_proj.geometry.x, y=pts_proj.geometry.y).to_xarray()

# a dataframe or series (for a single point)
pt_values_i = (
    merged.sel(pts_da[["x", "y"]], method="nearest").squeeze().compute().to_pandas()
)

if isinstance(pt_values_i, pd.Series):
    pt_values_i = pt_values_i.to_frame().transpose()
    pt_values_i.index = tdata.index

In [ ]:
len(pt_values_i)

In [ ]:
# Check columns of pt_values_i to see what's actually in there
print("pt_values_i columns:", pt_values_i.columns)

# If lulc_code is in tdata, check tdata length and nulls
print("tdata length:", len(tdata))
print("tdata nulls:", tdata['lulc_code'].isnull().sum())

# When merging, ensure you use a left join or check why rows are dropping:
# merged_df = pd.merge(tdata, pt_values_i, left_index=True, right_index=True, how='inner')

In [ ]:
# # Check how many NaNs exist per column in your extracted feature dataframe
# print(pt_values_i.isnull().sum())

# # Check how many rows remain if you dropna only on the target column vs all columns
# print("Valid targets:", pt_values_i['lulc_code'].dropna().shape[0])
# print("Valid complete rows:", dpt_values_i.dropna().shape[0])

In [ ]:
concat_df = pd.concat([tdata, pt_values_i], axis=1)

In [ ]:
# 1. See which specific columns have the most NaNs
print(concat_df.isnull().sum())

# 2. Clean properly: only drop rows where the target (lulc_code) is missing, 
# or fill/drop based strictly on key features instead of a blind table-wide dropna()
training_array = concat_df.dropna(subset=['lulc_code'])

# If certain raster features have NaNs, you can fill them or drop specifically for modeling:
# feature_cols = ['blue', 'green', 'red', 'ndvi', ...] # your feature list
# training_array = concat_df.dropna(subset=['lulc_code'] + feature_cols)

In [ ]:
# 1. Drop duplicate columns if pandas created suffixes or repeated coordinates (like 'y.1', 'x.1', etc.)
concat_df = concat_df.loc[:, ~concat_df.columns.duplicated()]

# 2. Verify columns and check where missing values are hiding
print("Cleaned columns:", concat_df.columns.tolist())
print("Null counts per column:\n", concat_df.isnull().sum())

# 3. Safely create your training array by dropping rows only where the target (lulc_code) is missing
training_array = concat_df.dropna(subset=['lulc_code'])

print("Final training array shape:", training_array.shape)

In [ ]:
training_array = pd.concat([tdata, pt_values_i], axis=1).to_crs(4326)
training_array = training_array.drop(
    columns=[
        "y",
        "x",
        "spatial_ref",
        "time",
        "fid",
        "index",
        "lulc_class",
        "path",
        "geometry",
        "layer",
        "id",
    ]
)
# Drop rows where there are any NaNs
training_array = training_array.dropna()

training_array.head()

In [ ]:
# # 1. Check feature names
# print("Model expected features:", getattr(model, "feature_names_in_", "None saved"))

# # 2. Check for constant or null features in training data
# for i, col in enumerate(training_array.columns if hasattr(training_array, "columns") else range(training_array.shape[1])):
#     vals = training_array[:, i] if not hasattr(training_array, "columns") else training_array[col]
#     print(f"Feature {col}: min={vals.min()}, max={vals.max()}, nans={np.isnan(vals).sum()}")

In [ ]:
training_array.to_csv("training_array.csv")

In [ ]:
classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=10,
    n_jobs=-1,
    random_state=42,
)

training_data = np.array(training_array)[:, 1:]
classes = np.array(training_array)[:, 0]

model = classifier.fit(training_data, classes)


In [ ]:
joblib.dump(model, "test_model.dump")

In [ ]:
# Print feature importances against column headings
fields_importances = sorted(
    zip(training_array.columns[1:], classifier.feature_importances_),
    key=lambda x: x[1],
    reverse=True,
)

for i in fields_importances:
    # Format as a table to 2 decinal places
    print(f"{i[0]:<11}| {i[1]:.2f}")
